In [ ]:
import json
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
import numpy as np
from pathlib import Path
from google.colab import drive
from collections import Counter

drive.mount('/content/drive')
PROJECT_DIR = Path('/content/drive/MyDrive')

def load_json(name):
    with open(PROJECT_DIR / name) as f:
        return json.load(f)

def load_pred_file(name):
    return {r['idx']: r for r in load_json(name) if 'idx' in r}

print("Loading raw prediction files...")
medqa = {
    'flan':     load_pred_file('flan_t5_results.json'),
    'llama':    load_pred_file('llama_results.json'),
    'qwen':     load_pred_file('qwen_results.json'),
    'gemma':    load_pred_file('gemma3n_results.json'),
    'deepseek': load_pred_file('deepseek_results.json'),
    'gpt4o':    load_pred_file('medqa_gpt4o_results.json'),
}
medmcqa = {
    'flan':  load_pred_file('medmcqa_flan_t5_results.json'),
    'llama': load_pred_file('medmcqa_llama_results.json'),
    'qwen':  load_pred_file('medmcqa_qwen_results.json'),
    'gemma': load_pred_file('medmcqa_gemma3n_results.json'),
    'gpt4o': load_pred_file('medmcqa_gpt4o_results.json'),
}
print(f"  MedQA:   {len(medqa)} models, n={len(medqa['flan'])}")
print(f"  MedMCQA: {len(medmcqa)} models, n={len(medmcqa['flan'])}")

print("\nLoading audit JSONs...")
audit = {
    'empirical_chance': load_json('audit_empirical_chance.json'),
    'frontier_sub':     load_json('audit_frontier_substitution.json'),
    'shuffle':          load_json('audit_shuffle_summary.json'),
    'pairwise':         load_json('audit_pairwise_lift_permutation.json'),
    'detector_mq':      load_json('detector_improved_validation.json'),
    'detector_mmc':     load_json('detector_medmcqa_validation.json'),
    'fivewrong':        load_json('audit_5wrong_baseline.json'),
    'non_dental':       load_json('audit_non_dental.json'),
    'test_retest':      load_json('audit_test_retest.json'),
    'triangulation':   load_json('triangulation_summary.json'),
}
for k in audit:
    print(f"  {k}")

def compute_unanim(data_dict, model_names):
    ids = sorted(data_dict['flan'].keys())
    n_subset = 0
    n_unanim = 0
    for i in ids:
        if not all(data_dict[m][i].get('correct') == 0 for m in model_names):
            continue
        preds = [data_dict[m][i].get('pred') for m in model_names]
        if not all(isinstance(p, str) and p in 'ABCD' for p in preds):
            continue
        n_subset += 1
        if len(set(preds)) == 1:
            n_unanim += 1
    return n_subset, n_unanim

print("\nComputing Figure 1 data (convergence per ensemble configuration)...")
fig1 = []
SPECS = [
    ('MedQA',   '3 mid-tier',       medqa,   ['llama','qwen','gemma']),
    ('MedQA',   '4 (incl DeepSeek)', medqa,   ['llama','qwen','gemma','deepseek']),
    ('MedQA',   '4 (incl GPT-4o)',   medqa,   ['llama','qwen','gemma','gpt4o']),
    ('MedQA',   '5 (both frontier)', medqa,   ['llama','qwen','gemma','deepseek','gpt4o']),
    ('MedMCQA', '4 strong',          medmcqa, ['llama','qwen','gemma','gpt4o']),
]
for dataset, label, data_dict, models in SPECS:
    n_subset, n_unanim = compute_unanim(data_dict, models)
    k = len(models)
    chance = (1/3) ** (k - 1)
    rate = n_unanim / max(1, n_subset)
    fig1.append({
        'dataset': dataset, 'label': label, 'k': k,
        'n_subset': n_subset, 'n_unanim': n_unanim,
        'rate': rate, 'chance_naive': chance,
        'ratio_naive': rate / chance,
    })
    print(f"  {dataset:<8} {label:<22} k={k}: "
          f"{n_unanim:>3}/{n_subset:<3} = {rate*100:>5.1f}%  "
          f"chance {chance*100:>5.2f}%  →  {rate/chance:>5.1f}x")

F2_CACHE = PROJECT_DIR / 'figure2_kintersection_data.json'
kdata = load_json('figure2_kintersection_data.json') if F2_CACHE.exists() else None
print(f"\nFigure 2 data: {'loaded from cache' if kdata else 'CACHE MISSING — rerun figure2 cell 1'}")

print(f"\nFigure 3 data:")
print(f"  Pairwise lifts: {len(audit['pairwise']['medqa'])} MedQA pairs, "
      f"{len(audit['pairwise']['medmcqa'])} MedMCQA pairs")
print(f"  Shuffle: {len(audit['shuffle']['per_model_under_shuffle'])} models")

def detector_points(det):
    total = sum(t['n'] for t in det['tier_results'].values())
    return {
        'route_pct': det['tier_results']['HIGH_RISK']['n'] / total,
        'recall':    det['high_risk_recall'],
        'precision': det['high_risk_precision'],
        'confident_wrong_rate': det['tier_results']['CONFIDENT'].get(
            'mid_wrong_rate', det['tier_results']['CONFIDENT'].get('wrong_rate')),
    }
fig4 = {
    'MedQA':   detector_points(audit['detector_mq']),
    'MedMCQA': detector_points(audit['detector_mmc']),
}
print(f"\nFigure 4 data: detector operating points")
for name, p in fig4.items():
    print(f"  {name:<8} routed {p['route_pct']*100:>4.1f}%, "
          f"recall {p['recall']*100:>4.1f}%, "
          f"precision {p['precision']*100:>4.1f}%, "
          f"confident-wrong {p['confident_wrong_rate']*100:>4.1f}%")

print(f"\n{'='*64}\nCROSS-CHECK against paper_numbers.json:\n{'='*64}")
paper = load_json('paper_numbers.json')
checks = [
    ('5-LLM MedQA rate',
     fig1[3]['rate'],
     paper['headline_5_llm']['unanimous_rate']),
    ('4-LLM MedQA rate (DeepSeek)',
     fig1[1]['rate'],
     paper['empirical_chance_4_model']['medqa']['observed_rate']),
    ('4-LLM MedMCQA rate',
     fig1[4]['rate'],
     paper['empirical_chance_4_model']['medmcqa']['observed_rate']),
    ('Detector MedQA precision',
     fig4['MedQA']['precision'],
     paper['detector_medqa']['high_risk_precision']),
    ('Detector MedMCQA precision',
     fig4['MedMCQA']['precision'],
     paper['detector_medmcqa']['high_risk_precision']),
]
all_ok = True
for label, computed, reference in checks:
    ok = abs(computed - reference) < 1e-4
    all_ok = all_ok and ok
    mark = "✓" if ok else "✗"
    print(f"  {mark} {label:<32} computed={computed*100:>5.2f}%  ref={reference*100:>5.2f}%")
print(f"\n{'ALL CHECKS PASS' if all_ok else 'MISMATCH — investigate before drawing'}")
print(f"\nNamespace ready: medqa, medmcqa, audit, fig1, kdata, fig4")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

def wilson_ci(k, n, conf=0.95):
    """Wilson score CI for a proportion."""
    if n == 0:
        return (0.0, 0.0)
    z = stats.norm.ppf(1 - (1 - conf) / 2)
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return (max(0, center - half), min(1, center + half))

medqa_primary_labels = ['3 mid-tier', '4 (incl DeepSeek)', '5 (both frontier)']
medqa_primary = [c for c in fig1
                 if c['dataset'] == 'MedQA' and c['label'] in medqa_primary_labels]
medqa_primary.sort(key=lambda c: c['k'])

medqa_gpt4o_k4 = next(c for c in fig1
                      if c['dataset'] == 'MedQA' and c['label'] == '4 (incl GPT-4o)')

medmcqa_k4 = next(c for c in fig1 if c['dataset'] == 'MedMCQA')

k_obs = np.array([c['k'] for c in medqa_primary])
rate_obs = np.array([c['rate'] for c in medqa_primary]) * 100
ci_obs = np.array([wilson_ci(c['n_unanim'], c['n_subset']) for c in medqa_primary]) * 100
ratios_obs = [c['ratio_naive'] for c in medqa_primary]

k_smooth = np.linspace(2.7, 5.3, 200)
chance_smooth = (1 / 3) ** (k_smooth - 1) * 100

fig, ax = plt.subplots(figsize=(9, 5.8), dpi=120)

C_OBS = '#1f3a68'
C_CHANCE = '#888888'
C_MMC = '#b8423f'

fig.text(0.5, 0.96,
         'Capable LLMs converge on the same wrong answer at rates that stay '
         'nearly flat as ensemble size grows,\n'
         'while independence chance drops geometrically — '
         'a 20× separation at k=5.',
         ha='center', va='top', fontsize=10.2, color='#222',
         linespacing=1.4, style='italic')

ax.plot(k_smooth, chance_smooth, color=C_CHANCE, linewidth=1.8,
        linestyle='--', label='Independence chance  (1/3)$^{k-1}$', zorder=3)

ax.fill_between(k_obs, ci_obs[:, 0], ci_obs[:, 1],
                color=C_OBS, alpha=0.15, zorder=2)
ax.plot(k_obs, rate_obs, color=C_OBS, linewidth=2.2,
        marker='o', markersize=9, markerfacecolor=C_OBS,
        markeredgecolor='white', markeredgewidth=1.2,
        label='Observed (MedQA-USMLE)', zorder=5)

ax.scatter([medqa_gpt4o_k4['k'] + 0.13], [medqa_gpt4o_k4['rate'] * 100],
           s=70, facecolor='white', edgecolor=C_OBS, linewidth=1.5,
           marker='o', zorder=6,
           label='MedQA k=4 (GPT-4o substitution, robustness)')

ax.scatter([medmcqa_k4['k'] - 0.13], [medmcqa_k4['rate'] * 100],
           s=110, color=C_MMC, marker='D', edgecolor='white', linewidth=1.2,
           zorder=6, label='MedMCQA k=4 (cross-dataset replication)')

k5_obs_y = rate_obs[-1]
k5_chance_y = (1/3) ** (k_obs[-1] - 1) * 100
ax.annotate('', xy=(k_obs[-1] + 0.27, k5_obs_y),
            xytext=(k_obs[-1] + 0.27, k5_chance_y),
            arrowprops=dict(arrowstyle='<->', color='black', lw=2.0))
ax.text(k_obs[-1] + 0.36, np.sqrt(k5_obs_y * k5_chance_y),
        f'{ratios_obs[-1]:.1f}×', fontsize=18, fontweight='bold',
        color='black', va='center', ha='left')

for k, rate, c in zip(k_obs, rate_obs, medqa_primary):
    ax.text(k, rate * 1.35, f'{c["n_unanim"]}/{c["n_subset"]}',
            ha='center', va='bottom', fontsize=8.5, color='#444')

ax.set_yscale('log')
ax.set_xlim(2.5, 5.65)
ax.set_ylim(0.5, 60)
ax.set_xticks([3, 4, 5])
ax.set_yticks([1, 3, 10, 30])
ax.set_yticklabels(['1%', '3%', '10%', '30%'])
ax.set_xlabel('Ensemble size k  (number of strong LLMs)', fontsize=10.5)
ax.set_ylabel('Unanimous-wrong rate  (log scale)', fontsize=10.5)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', linestyle=':', alpha=0.35, zorder=0)

ax.legend(loc='lower left', frameon=False, fontsize=8.5, handlelength=2.4)

plt.tight_layout(rect=[0, 0, 1, 0.92])

out_png = PROJECT_DIR / 'figure1_divergence.png'
out_pdf = PROJECT_DIR / 'figure1_divergence.pdf'
plt.savefig(out_png, dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(out_pdf, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {out_png.name}, {out_pdf.name}')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

DATASETS = [
    ('medqa',   'MedQA-USMLE (n=1,273)',   '#1f3a68', 'o'),
    ('medmcqa', 'MedMCQA (n=2,816)',       '#b8423f', 'D'),
]

fig, ax = plt.subplots(figsize=(9, 5.6), dpi=120)

fig.text(0.5, 0.965,
         'Bimodal correlated-failure structure: convergence is concentrated at the extremes '
         '(k=0 and k=max),\nwith intermediate k depleted — replicated across both datasets.',
         ha='center', va='top', fontsize=10.2, color='#222',
         linespacing=1.4, style='italic')

ax.axhline(0, color='#888', linewidth=1.0, zorder=1)
for level, alpha in [(2, 0.06), (3, 0.10)]:
    ax.axhspan(-level, level, color='#bbbbbb', alpha=alpha, zorder=0)
ax.text(4.18, 2.05, '|z|=2', fontsize=8, color='#777', va='bottom', ha='left')
ax.text(4.18, 3.05, '|z|=3', fontsize=8, color='#555', va='bottom', ha='left')

for key, label, color, marker in DATASETS:
    rows = kdata[key]['rows']
    ks = np.array([r['k'] for r in rows])
    zs = np.array([r['z'] for r in rows])
    ax.plot(ks, zs, color=color, linewidth=2.0, alpha=0.85, zorder=3)
    ax.scatter(ks, zs, s=110, color=color, marker=marker,
               edgecolor='white', linewidth=1.4, zorder=4, label=label)

    for xi, z in zip(ks, zs):
        if abs(z) >= 5:
            offset = 0.9 if z > 0 else -0.9
            va = 'bottom' if z > 0 else 'top'
            ax.text(xi, z + offset, f'{z:+.1f}',
                    ha='center', va=va, fontsize=9,
                    color=color, fontweight='bold')

ax.set_xlabel('k = strong models wrong  |  Flan correct', fontsize=10.5)
ax.set_ylabel('Deviation from independence null  (z-score)', fontsize=10.5)
ax.set_xticks([0, 1, 2, 3, 4])
ax.set_xlim(-0.4, 4.4)

zmax = max(abs(r['z']) for ds in DATASETS for r in kdata[ds[0]]['rows'])
ax.set_ylim(-zmax * 1.25, zmax * 1.25)

ax.text(-0.32, zmax * 0.62, 'Excess\nconvergence', fontsize=8.5,
        color='#222', va='top', ha='left', style='italic', alpha=0.8)
ax.text(-0.32, -zmax * 0.62, 'Depletion\n(below chance)', fontsize=8.5,
        color='#222', va='bottom', ha='left', style='italic', alpha=0.8)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', linestyle=':', alpha=0.30, zorder=0)
ax.legend(loc='center right', frameon=False, fontsize=9.5)

plt.tight_layout(rect=[0, 0, 1, 0.92])

out_png = PROJECT_DIR / 'figure2_bimodality.png'
out_pdf = PROJECT_DIR / 'figure2_bimodality.pdf'
plt.savefig(out_png, dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(out_pdf, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {out_png.name}, {out_pdf.name}')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import TwoSlopeNorm

MODEL_ORDER = {
    'medqa':   ['flan', 'llama', 'qwen', 'gemma', 'deepseek'],
    'medmcqa': ['flan', 'llama', 'qwen', 'gemma', 'gpt4o'],
}
PRETTY = {
    'flan': 'Flan-T5\n(near-random)', 'llama': 'Llama-3-8B',
    'qwen': 'Qwen-2.5-7B', 'gemma': 'Gemma-3n-E4B',
    'deepseek': 'DeepSeek-V3', 'gpt4o': 'GPT-4o',
}

def build_lift_matrix(pairwise_rows, order):
    """Build symmetric N×N lift matrix from list of pair-records."""
    n = len(order)
    M = np.full((n, n), np.nan)
    np.fill_diagonal(M, 1.0)
    idx = {m: i for i, m in enumerate(order)}
    for r in pairwise_rows:
        a, b = r['pair'].split('_')
        i, j = idx[a], idx[b]
        M[i, j] = r['observed']
        M[j, i] = r['observed']
    return M

fig, axes = plt.subplots(1, 2, figsize=(13, 5.6), dpi=120)

fig.text(0.5, 0.965,
         'Capable LLMs share specific failure modes: pairwise lift = P(B wrong | A wrong) / P(B wrong). '
         'Lift > 1 indicates\ncorrelated failures. Mid-tier × mid-tier block elevated; '
         'Flan row/column near independence.',
         ha='center', va='top', fontsize=10.0, color='#222',
         linespacing=1.4, style='italic')

vmin, vmax = 0.95, 1.80
norm = TwoSlopeNorm(vmin=vmin, vcenter=1.0, vmax=vmax)
cmap = plt.get_cmap('RdBu_r')

for ax, (key, title) in zip(axes, [('medqa', 'MedQA-USMLE'),
                                    ('medmcqa', 'MedMCQA')]):
    order = MODEL_ORDER[key]
    M = build_lift_matrix(audit['pairwise'][key], order)

    im = ax.imshow(M, cmap=cmap, norm=norm, aspect='equal')

    for i in range(len(order)):
        for j in range(len(order)):
            val = M[i, j]
            if np.isnan(val):
                continue

            text_color = 'white' if (val > 1.55 or val < 0.85) else '#111'
            weight = 'bold' if i != j else 'normal'
            ax.text(j, i, f'{val:.2f}',
                    ha='center', va='center',
                    fontsize=10.5, color=text_color, fontweight=weight)

    ax.set_xticks(range(len(order)))
    ax.set_yticks(range(len(order)))
    ax.set_xticklabels([PRETTY[m] for m in order], fontsize=8.5, rotation=30, ha='right')
    ax.set_yticklabels([PRETTY[m] for m in order], fontsize=8.5)
    ax.set_title(title, fontsize=11.5, fontweight='bold', pad=10)

    from matplotlib.patches import Rectangle
    ax.add_patch(Rectangle((0.5, 0.5), 3, 3, fill=False,
                            edgecolor='black', linewidth=2.0, zorder=5))

cax = fig.add_axes([0.30, 0.04, 0.40, 0.025])
cb = fig.colorbar(im, cax=cax, orientation='horizontal')
cb.set_label('Pairwise failure lift  (1.0 = independence)', fontsize=9.5)
cb.ax.tick_params(labelsize=8.5)

plt.tight_layout(rect=[0, 0.10, 1, 0.91])

out_png = PROJECT_DIR / 'figure3_pairwise_lift.png'
out_pdf = PROJECT_DIR / 'figure3_pairwise_lift.pdf'
plt.savefig(out_png, dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(out_pdf, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {out_png.name}, {out_pdf.name}')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

DATASETS = [
    ('MedQA',   '#1f3a68', 'o'),
    ('MedMCQA', '#b8423f', 'D'),
]

fig, ax = plt.subplots(figsize=(8.5, 6.0), dpi=120)

fig.text(0.5, 0.965,
         'A simple routing rule — flag questions where mid-tier LLMs unanimously agree '
         'but the frontier model\ndisagrees — catches a meaningful fraction of silent '
         'ensemble failures at low human-review cost.',
         ha='center', va='top', fontsize=10.2, color='#222',
         linespacing=1.4, style='italic')

xs = np.linspace(0, 20, 100)
ax.plot(xs, xs, color='#888', linewidth=1.4, linestyle='--',
        label='Random routing (baseline)', zorder=2)
ax.fill_between(xs, 0, xs, color='#cccccc', alpha=0.15, zorder=1)

for name, color, marker in DATASETS:
    p = fig4[name]
    x = p['route_pct'] * 100
    y = p['recall'] * 100
    prec = p['precision'] * 100

    ax.scatter(x, y, s=240, color=color, marker=marker,
               edgecolor='white', linewidth=2.0, zorder=5,
               label=f'{name}  (precision = {prec:.1f}%)')

    label_xy = (x + 1.2, y + 1.5)
    ax.annotate(
        f'{name}\nroute {x:.1f}%, catch {y:.1f}%\nprecision {prec:.1f}%',
        xy=(x, y), xytext=label_xy,
        fontsize=9, color=color, fontweight='bold',
        ha='left', va='bottom',
        arrowprops=dict(arrowstyle='-', color=color, lw=0.8, alpha=0.5))

for name, color, marker in DATASETS:
    p = fig4[name]
    x = p['route_pct'] * 100
    y = p['recall'] * 100
    ax.plot([x, x], [0, y], color=color, linewidth=0.6, alpha=0.4, linestyle=':', zorder=1)
    ax.plot([0, x], [y, y], color=color, linewidth=0.6, alpha=0.4, linestyle=':', zorder=1)

ax.set_xlabel('Questions routed to human review  (%)', fontsize=10.5)
ax.set_ylabel('Silent ensemble errors caught  (% recall)', fontsize=10.5)

ax.set_xlim(0, 12)
ax.set_ylim(0, 25)
ax.set_xticks([0, 2, 4, 6, 8, 10, 12])
ax.set_yticks([0, 5, 10, 15, 20, 25])

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(linestyle=':', alpha=0.30, zorder=0)
ax.legend(loc='lower right', frameon=False, fontsize=9.5)

inset = fig.add_axes([0.18, 0.60, 0.18, 0.20])
conf_vals = [fig4[d[0]]['confident_wrong_rate'] * 100 for d in DATASETS]
inset.bar(range(len(DATASETS)), conf_vals,
          color=[d[1] for d in DATASETS],
          edgecolor='black', linewidth=0.6)
for i, v in enumerate(conf_vals):
    inset.text(i, v + 0.4, f'{v:.1f}%', ha='center', va='bottom',
               fontsize=8.5, fontweight='bold')
inset.set_xticks(range(len(DATASETS)))
inset.set_xticklabels([d[0] for d in DATASETS], fontsize=8)
inset.set_ylabel('Wrong rate (%)', fontsize=8)
inset.set_title('CONFIDENT tier\n(all 4 models agree)',
                fontsize=8.5, pad=5)
inset.set_ylim(0, max(conf_vals) * 1.5)
inset.spines['top'].set_visible(False)
inset.spines['right'].set_visible(False)
inset.tick_params(axis='y', labelsize=7.5)

plt.subplots_adjust(top=0.86, bottom=0.10, left=0.10, right=0.96)

out_png = PROJECT_DIR / 'figure4_detector.png'
out_pdf = PROJECT_DIR / 'figure4_detector.pdf'
plt.savefig(out_png, dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(out_pdf, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {out_png.name}, {out_pdf.name}')